# TEST-07 POSTFIX: Cost-Drag Denominator

WP-5b preserves the reported cost-drag fraction while fixing Gate 5 pass/fail behavior for non-positive gross PnL.

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
from lib.backtest import BacktestResult
from lib.evaluation import check_promotion_gates

def gate5_result(gross, cost):
    idx = pd.date_range('2020-01-01', periods=8, freq='1h', tz='UTC')
    eq = pd.Series([10_000.0, 10_010.0, 10_020.0, 10_030.0, 10_040.0, 10_050.0, 10_060.0, 10_070.0], index=idx, name='equity')
    res = BacktestResult(
        equity_curve=eq,
        trades_df=pd.DataFrame(columns=['entry_bar','exit_bar','direction','entry_price','exit_price','gross_pnl','fees','slippage','net_pnl']),
        per_bar_pnl=eq.diff().fillna(0.0).rename('pnl'),
        realized_pnl=float(gross),
        unrealized_pnl=0.0,
        fees_paid=float(cost),
        slippage_paid=0.0,
        funding_paid=0.0,
        skipped_bars=0,
        skipped_bar_timestamps=[],
        params={},
    )
    report = check_promotion_gates(res, trial_count=1, params_meta={'n_free_params': 1, 'sensitivity_min_ratio': 1.0, 'is_sharpe': 1.0}, freq='1h')
    g5 = {g.gate_id: g for g in report.gates}[5]
    pre_drag = float(cost / gross) if gross != 0 else 0.0
    pre_pass = pre_drag <= 0.50
    return pre_drag, pre_pass, g5

cases = [('positive gross', 1000.0, 100.0), ('zero gross', 0.0, 0.0), ('negative gross', -1000.0, 100.0)]
print('=== Gate 5 pre-fix vs post-fix ===')
for name, gross, cost in cases:
    pre_drag, pre_pass, g5 = gate5_result(gross, cost)
    print(f'{name:15s} gross={gross:8.2f} cost={cost:7.2f} pre_value={pre_drag:7.4f} '
          f'pre_pass={str(pre_pass):5s} post_value={g5.value:7.4f} '
          f'post_pass={str(g5.passed):5s} note={g5.note}')


=== Gate 5 pre-fix vs post-fix ===
positive gross  gross= 1000.00 cost= 100.00 pre_value= 0.1000 pre_pass=True  post_value= 0.1000 post_pass=True  note=
zero gross      gross=    0.00 cost=   0.00 pre_value= 0.0000 pre_pass=True  post_value= 0.0000 post_pass=False note=gross_pnl=0, cost drag undefined
negative gross  gross=-1000.00 cost= 100.00 pre_value=-0.1000 pre_pass=True  post_value=-0.1000 post_pass=False note=gross_pnl=-1000.00 <= 0: cost drag undefined/misleading


In [ ]:
positive = gate5_result(1000.0, 100.0)[2]
zero = gate5_result(0.0, 0.0)[2]
negative = gate5_result(-1000.0, 100.0)[2]
verdict = 'PASS' if (positive.value == 0.1 and positive.passed and
                     zero.value == 0.0 and not zero.passed and
                     negative.value == -0.1 and not negative.passed) else 'FAIL'
print(f'=== VERDICT: {verdict} ===')
print('gross_pnl > 0: reported value and pass/fail convention are unchanged.')
print('gross_pnl == 0: reported value remains 0.0, but Gate 5 now fails.')
print('gross_pnl < 0: reported negative diagnostic value is preserved, but Gate 5 now fails.')
print('Reported value scale remains a fraction; threshold remains 0.50.')


=== VERDICT: PASS ===
gross_pnl > 0: reported value and pass/fail convention are unchanged.
gross_pnl == 0: reported value remains 0.0, but Gate 5 now fails.
gross_pnl < 0: reported negative diagnostic value is preserved, but Gate 5 now fails.
Reported value scale remains a fraction; threshold remains 0.50.
